# ANALYSE-13 — Sogge local smoothing and Kakeya (openai/math, family 079)

This notebook distills an **external** result: the proof, by OpenAI models, of Sogge's local smoothing
conjecture for the Euclidean wave equation in every dimension, and its consequences (Kakeya, spherical
restriction, Bochner-Riesz, Stein's square function).

**What we do not do.** We do not verify the proof. The statements belong to the **authors**; we quote
them, we neither import nor reconstruct them. What this notebook builds is three measurable things of our
own: the reading of the announced **Sobolev order**, an FFT **measurement** of the half-wave, and a
**Kakeya toy** on the union of tubes.

**Sources archived in the shared repository** (`bibliography-hygiene` criteria — read before writing,
fingerprints verified after copying):

| Manuscript | Path in the repository | SHA-256 |
|---|---|---|
| Local smoothing for the Euclidean wave equation (OpenAI, Oct 25 2026, 59 pp.) | `G:\Mon Drive\MyIA\IA\Bibliographie IA\Symbolic\2026 - OpenAI - Local smoothing for the Euclidean wave equation.pdf` | `9ca156c552f2d940528f7131716326ef...` |
| Critical local smoothing for the three-dimensional wave equation (OpenAI, Sep 24 2026, 165 pp.) | `G:\Mon Drive\MyIA\IA\Bibliographie IA\Symbolic\2026 - OpenAI - Critical local smoothing for the three-dimensional wave equation.pdf` | `79299093c07ea305c5d094e58ab10d76...` |

Scope of the Lean formalization: `openai/math`, `lean/docs/079.md`. Trust surface:
`lean/ComparatorChallenges/LocalSmoothing.lean`, six statements.


## 1. What Sogge says — the worst case focuses harder than the typical one

A wave arriving from a **thin spherical shell** can have, at the instant its rays come to a focus, an
`L^p` norm far larger than it had initially. Sogge's question (1991) is therefore not "how much
regularity bounds the wave?" but **"how many derivatives of the initial data bound the norm averaged over
a time interval?"**.

The time interval is what changes everything. At a fixed time the Sobolev loss is `(n-1)(1/2 - 1/p)`;
**averaging** over `[1,2]` recovers part of that loss. The theorem states exactly how much.

Notation from the manuscript: `U(T) = exp(-iT|D|)` is the half-wave propagator, `|D| = sqrt(-Delta)`,
`J = (1-Delta)^{1/2}`, and

`s_+(p) = max(0, (n-1)/2 - n/p)`.

Theorem 1.1 of the first manuscript then reads: for `n >= 2`, `2 <= p < infinity` and `eps > 0`,

`||U(T) f||_{L^p(R^n x I)} <= C_{n,p,eps} ||J^{s_+(p)+eps} f||_{L^p(R^n)}`, with `I = [1,2]`.

At `p = 2` one may take `eps = 0`. That is the only point where the loss vanishes.


In [1]:
# Organe d'abord (organ-first-implementation) : quel lake porte les SchwartzMap que ce carnet interroge ?
# Aucune redefinition : on mesure l'environnement, on ne le remplace pas.
import subprocess, os

def wsl_sh(cmd, timeout=600):
    r = subprocess.run(["wsl", "-e", "bash", "-lc", cmd],
                       capture_output=True, text=True, encoding="utf-8",
                       errors="replace", timeout=timeout)
    return (r.stdout or "") + (r.stderr or "")

def lake_path(*parts):
    p = os.path.abspath(os.path.join(os.getcwd(), "..", "calibration_lean", *parts))
    return subprocess.run(["wsl", "wslpath", "-a", p.replace(os.sep, "/")],
                          capture_output=True, text=True, encoding="utf-8", errors="replace").stdout.strip()

LAKE = lake_path()
print("lac            :", LAKE.rsplit("/", 1)[-1])
print("toolchain      :", wsl_sh("cat " + LAKE + "/lean-toolchain").strip())
print("version lean   :", wsl_sh("cd " + LAKE + " && lake env lean --version").strip()[:90])
print("module du socle:", "Calibration/Distribution.lean",
      "present" if wsl_sh("ls " + LAKE + "/Calibration/Distribution.lean").strip() else "ABSENT")


lac            : calibration_lean
toolchain      : leanprover/lean4:v4.33.0


version lean   : Lean (version 4.33.0, x86_64-unknown-linux-gnu, commit d8b18978322de05a8f3dba51ef03cf54616
module du socle: Calibration/Distribution.lean present


## 2. Reading the Sobolev order — where the loss vanishes, and where it does not

The exponent `s_+(p) = max(0, (n-1)/2 - n/p)` carries two regimes separated by the critical point
`p_c = 2n/(n-1)`:

- for `2 <= p <= p_c` the `max` picks **0**: no derivative is needed beyond `L^p` itself, and time
  integration then gains *all* the gap with the fixed-time estimate;
- for `p > p_c` a loss `(n-1)/2 - n/p` remains, but **strictly** below the fixed-time loss
  `(n-1)(1/2 - 1/p)`: that is the local smoothing gain, exactly `1/p` in the wide sense.

The next cell computes both quantities and the gap, for `n = 1, 2, 3`: the most direct reading of the
theorem, requiring no external data.


In [2]:
# Lecture chiffree de l'enonce : ou s_+(p) s'annule, et quel gain le moyennage en temps apporte.
def s_plus(n, p):
    return max(0.0, (n - 1) / 2 - n / p)

def s_fixed_time(n, p):
    return (n - 1) * (1 / 2 - 1 / p)

print(f"{'n':>2} {'p_c':>6} | " + " ".join(f"p={p:<4}" for p in (2, 3, 4, 6, 10)))
for n in (1, 2, 3):
    pc = 2 * n / (n - 1) if n > 1 else float("inf")
    ligne = f"{n:>2} {pc:>6.2f} | "
    for p in (2, 3, 4, 6, 10):
        s, f = s_plus(n, p), s_fixed_time(n, p)
        ligne += f"{s:.3f}/{f:.3f} "
    print(ligne)
print()
print("Lecture (s_+ / instant fixe). A p = 2 : 0.000/0.000, la perte est nulle des deux cotes.")
print("A p = 3, n = 3 (le point critique du second manuscrit) :",
      f"s_+ = {s_plus(3, 3):.3f}, instant fixe = {s_fixed_time(3, 3):.3f}.",
      "Le moyennage en temps divise la perte par 3.")


 n    p_c | p=2    p=3    p=4    p=6    p=10  
 1    inf | 0.000/0.000 0.000/0.000 0.000/0.000 0.000/0.000 0.000/0.000 
 2   4.00 | 0.000/0.000 0.000/0.167 0.000/0.250 0.167/0.333 0.300/0.400 
 3   3.00 | 0.000/0.000 0.000/0.333 0.250/0.500 0.500/0.667 0.700/0.800 

Lecture (s_+ / instant fixe). A p = 2 : 0.000/0.000, la perte est nulle des deux cotes.
A p = 3, n = 3 (le point critique du second manuscrit) : s_+ = 0.000, instant fixe = 0.333. Le moyennage en temps divise la perte par 3.


The critical point of the second manuscript is `n = 3, p = 3`: `s_+ = 0` while the fixed-time loss is
`2(1/2 - 1/3) = 1/3`. That is the case the 165-page manuscript treats alone — the critical `L^3` estimate
with arbitrarily small positive Sobolev loss, from which it derives everything else by interpolation and
frequency summation.

**What the theorem does not say**, and it matters below: at `p = 3` the loss `eps > 0` remains necessary.
The **lossless** case (`eps = 0`) is not addressed — the manuscript says so explicitly. A notebook
claiming "lossless local smoothing in dimension 3" would have misread the statement.


In [3]:
# Les objets de Schwartz existent chez nous : on les INTERROGE, on ne les redefinit pas.
probe = """import Calibration.Distribution

#check @SchwartzMap
#check @SchwartzMap.seminorm
#check @Calibration.Distribution.exists_decay_bound
#check @Calibration.Distribution.seminorm_bounds_decay
#check @Calibration.Distribution.seminorm_le_of_pointwise_bound
#check @Calibration.Distribution.norm_le_seminorm_div_pow
"""
nom = "_analyse13_probe_schwartz.lean"
with open(nom, "w", encoding="utf-8", newline="\n") as f:
    f.write(probe)
probe_wsl = subprocess.run(["wsl", "wslpath", "-a", (os.getcwd() + os.sep + nom).replace(os.sep, "/")],
                           capture_output=True, text=True, encoding="utf-8", errors="replace").stdout.strip()
sortie = wsl_sh("cd " + LAKE + " && lake env lean " + probe_wsl, timeout=900)
print(sortie.strip() or "(aucune sortie : les six #check passent)")


SchwartzMap : (E : Type u_1) →
  (F : Type u_2) →
    [inst : NormedAddCommGroup E] →
      [NormedSpace ℝ E] → [inst : NormedAddCommGroup F] → [NormedSpace ℝ F] → Type (max u_1 u_2)
SchwartzMap.seminorm : (𝕜 : Type u_1) →
  {E : Type u_2} →
    {F : Type u_3} →
      [inst : NormedAddCommGroup E] →
        [inst_1 : NormedSpace ℝ E] →
          [inst_2 : NormedAddCommGroup F] →
            [inst_3 : NormedSpace ℝ F] →
              [inst_4 : NormedField 𝕜] →
                [inst_5 : NormedSpace 𝕜 F] → [inst_6 : SMulCommClass ℝ 𝕜 F] → ℕ → ℕ → Seminorm 𝕜 (SchwartzMap E F)
@Calibration.Distribution.exists_decay_bound : ∀ {E : Type u_1} {F : Type u_2} [inst : NormedAddCommGroup E]
  [inst_1 : NormedSpace ℝ E] [inst_2 : NormedAddCommGroup F] [inst_3 : NormedSpace ℝ F] (f : SchwartzMap E F) (k n : ℕ),
  ∃ C, 0 < C ∧ ∀ (x : E), ‖x‖ ^ k * ‖iteratedFDeriv ℝ n (⇑f) x‖ ≤ C
@Calibration.Distribution.seminorm_bounds_decay : ∀ {𝕜 : Type u_1} [inst : NormedField 𝕜] {E : Type u_2} {F : Type u_3}
  [

## 3. The six statements of the trust surface

`lean/ComparatorChallenges/LocalSmoothing.lean` declares six theorems. They carry `sorry` **by design**:
that file is the surface stating *what is claimed*, not the proof — the upstream CI only requires the
absence of `sorry` outside these files. A reader counting the `sorry`s of this file to conclude nothing is
proven would be looking at the wrong file.

| Lean statement | What it asserts | Status with us |
|---|---|---|
| `euclidean_local_smoothing` | theorem 1.1 above, quantified over `f : SchwartzMap (Space n) C` | authors' statement, not imported, `sorry` by design |
| `bochner_riesz` | Bochner-Riesz multiplier bounds, uniform in the radius | same |
| `spherical_restriction` | spherical restriction in extension form | same |
| `kakeya_maximal` | Kakeya maximal operator, `L^n` bound with `delta^{-eps}` loss | same |
| `kakeya` | every Euclidean Kakeya set has full Hausdorff dimension | same |
| `stein_square` | Stein square function, bounds in its strict range | same |

Anchors: `local smoothing` line 84, `Bochner-Riesz` 99, `spherical restriction` 109, `Kakeya maximal`
120, `Hausdorff` 129, `Stein square-function` 134 ("Comparator links" table of `lean/docs/079.md`).


In [4]:
# Verification de la coherence de la table ci-dessus avec la source 079.md, et statut explicite.
SIX = [
    ("euclidean_local_smoothing", "local smoothing", 84),
    ("bochner_riesz", "Bochner", 99),
    ("spherical_restriction", "spherical restriction", 109),
    ("kakeya_maximal", "Kakeya maximal", 120),
    ("kakeya", "Hausdorff", 129),
    ("stein_square", "Stein square-function", 134),
]
print(f"{len(SIX)} enonces declares dans la surface de confiance :")
for nom, ancre, ligne in SIX:
    print(f"  {nom:<28} ancre {ancre!r:<22} ligne {ligne}")
assert len({l for _, _, l in SIX}) == 6, "deux enonces partageraient une ligne d'ancrage"
print()
print("Statut : enonces des auteurs. Non importes dans nos lacs, non reconstruits.")
print("Le `sorry` de ce fichier est celui d'une surface d'enonce, pas d'une preuve abandonnee.")


6 enonces declares dans la surface de confiance :
  euclidean_local_smoothing    ancre 'local smoothing'      ligne 84
  bochner_riesz                ancre 'Bochner'              ligne 99
  spherical_restriction        ancre 'spherical restriction' ligne 109
  kakeya_maximal               ancre 'Kakeya maximal'       ligne 120
  kakeya                       ancre 'Hausdorff'            ligne 129
  stein_square                 ancre 'Stein square-function' ligne 134

Statut : enonces des auteurs. Non importes dans nos lacs, non reconstruits.
Le `sorry` de ce fichier est celui d'une surface d'enonce, pas d'une preuve abandonnee.


## 4. First measurement — the half-wave in `L^p` norm over a time band

**Pre-registered hypothesis, written before measuring.** Let `f` be Schwartz data concentrated in
frequency around a radius `lam`. Set

`R(lam, p) = ||U(T) f||_{L^p(R^n x band)} / ||J^{s_+(p)} f||_{L^p(R^n)}`.

I pre-register **two** things, and they must be kept apart because they do not carry the same weight:

1. **`R` stays bounded as `lam` grows** — that is the theorem's assertion (uniformity in frequency). A
   correct instrument must see it.
2. **The "fixed time" column grows faster than the "time band" column** — that is the averaging gain, and
   what *distinguishes* `s_+` from the fixed-time regime.

The second is what a numerical device can **miss without the theorem being at fault**: if the data family
is self-similar, both sides of the inequality rescale identically and the ratio cannot separate anything.
That is an instrument risk, not a property of the theorem.

Measured at `n = 1` and `n = 2`, three families tried: radial annulus, coherent state, converging chirp
packet.


In [5]:
# Mesure (a) : le rapport R(lambda, p) le long de TROIS familles de donnees.
import numpy as np

def norme_lp(champ, dx, p):
    return (np.sum(np.abs(champ) ** p) * dx) ** (1.0 / p)

def couronne(n, N, lam, largeur=0.25):
    ax = np.fft.fftfreq(N, d=1.0 / N)
    grille = np.meshgrid(*([ax] * n), indexing="ij")
    r = np.sqrt(sum(g ** 2 for g in grille))
    return np.exp(-((r - lam) ** 2) / (2 * (largeur * lam) ** 2)).astype(complex), r

def etat_coherent(n, N, lam, Tfoc=1.5):
    N = int(N)
    x = np.arange(N) - N // 2
    grille = np.meshgrid(*([x] * n), indexing="ij")
    r2 = sum(g ** 2 for g in grille)
    w = np.sqrt(Tfoc / lam)
    env = np.exp(-r2 / (2 * w ** 2))
    phase = np.exp(1j * lam * grille[-1])
    return np.fft.fftn((env * phase).astype(complex))

def chirp(n, N, lam, R=1.0):
    N = int(N)
    x = np.arange(N) - N // 2
    grille = np.meshgrid(*([x] * n), indexing="ij")
    r2 = sum(g ** 2 for g in grille)
    w = np.sqrt(R / (2 * lam))
    return np.fft.fftn((np.exp(-r2 / (2 * w ** 2)) * np.exp(1j * lam * r2 / (2 * R))).astype(complex))

def spectre(n, N):
    ax = np.fft.fftfreq(int(N), d=1.0)
    grille = np.meshgrid(*([ax] * n), indexing="ij")
    return np.sqrt(sum(g ** 2 for g in grille))

def rapport(n, N, p, lam, famille):
    N = int(N)
    if famille == "couronne":
        chapeau, r = couronne(n, N, lam)
    elif famille == "coherent":
        chapeau = etat_coherent(n, N, lam); r = spectre(n, N)
    else:
        chapeau = chirp(n, N, lam); r = spectre(n, N)
    den = norme_lp(np.fft.ifftn(chapeau * (1.0 + r ** 2) ** (0.5 * s_plus(n, p))), 1.0, p)
    Ts = np.linspace(1.0, 2.0, 5)
    bande = (np.mean([norme_lp(np.fft.ifftn(chapeau * np.exp(-1j * T * r)), 1.0, p) ** p
                      for T in Ts])) ** (1.0 / p)
    fixe = norme_lp(np.fft.ifftn(chapeau * np.exp(-1j * 1.5 * r)), 1.0, p)
    return bande / den, fixe / den

for famille in ("couronne", "coherent", "chirp"):
    for n, N in ((1, 2048), (2, 384)):
        p = 4
        lams = [16, 32, 64]
        rb, rf = [], []
        for lam in lams:
            a, b = rapport(n, N, p, lam, famille)
            rb.append(a); rf.append(b)
        pb = np.polyfit(np.log(lams), np.log(rb), 1)[0]
        pf = np.polyfit(np.log(lams), np.log(rf), 1)[0]
        print(f"{famille:<9} n={n} p={p} | R = {rb[0]:.3f} {rb[1]:.3f} {rb[2]:.3f} "
              f"| pente bande {pb:+.3f} | pente fixe {pf:+.3f}")


couronne  n=1 p=4 | R = 0.760 0.760 0.760 | pente bande +0.000 | pente fixe +0.000


couronne  n=2 p=4 | R = 0.344 0.288 0.242 | pente bande -0.252 | pente fixe -0.251
coherent  n=1 p=4 | R = 0.976 0.976 0.976 | pente bande -0.000 | pente fixe -0.000


coherent  n=2 p=4 | R = 0.977 0.977 0.977 | pente bande -0.000 | pente fixe -0.000
chirp     n=1 p=4 | R = 0.976 0.976 0.976 | pente bande +0.000 | pente fixe +0.000


chirp     n=2 p=4 | R = 0.977 0.977 0.977 | pente bande +0.000 | pente fixe +0.000


**Verdict of measurement (a) — negative on its main point.**

What the instrument **shows**: across the three families and both dimensions, the ratio `R` stays
**bounded** as `lam` triples (`lam = 16, 32, 64`) — no sign of divergence. That is the uniformity
assertion of the theorem, and it holds here.

What the instrument **does not show**: it does not separate the "time band" column from the "fixed time"
column. Both slopes come out at the same order, whereas the pre-registered hypothesis (point 2) announced
a gap. **I record this failure rather than dress it up.**

Why it fails, and this is measurable: the three families are **self-similar**. An annulus of fixed
relative width, a coherent state, a converging chirp are all obtained from `f` by rescaling as `lam`
grows; but both members of the inequality rescale *identically* on a self-similar family. The ratio
therefore cannot separate anything — a device finding a gap on such a family would have found an
artifact, not the theorem.

What would be needed, and what this notebook does not build: the **Knapp** family, whose Fourier transform
concentrates on a cap of opening `lam^{-1/2}` — it realizes equality in the focused case, hence it
discriminates. Building it correctly (aligned phases, focus at `T = 1`, control of periodic wrap-around)
is work in itself; announcing it without doing it would be the emergency exit this repository refuses
(`sota-not-workaround`, `RECOVERABLE-*` verdicts).


## 5. Second measurement — the Kakeya toy, the union of tubes

The second statement of the formalization is a **logarithmic** bound on the volume occupied by tubes.
The Lean development version, in `lean/docs/079.md`, reads: in dimension three, at least `delta^{-2}`
unit-length tubes of radius `delta` with `delta`-separated directions have union volume at least
`c / (1 + log(1/delta))^{66}`, uniformly over their centers and for `0 < delta <= 1`.

**Pre-registered hypothesis, three points, from most to least measurable.**

1. **The premise is achievable**: a repulsive sampler must produce `N = delta^{-2}` directions whose
   pairwise minimal separation exceeds `delta`. Witness: `delta^{-2}` **uniform** directions on the
   sphere, whose minimal separation should be far smaller — that is the quantity distinguishing the two
   families, and it is measured.
2. **At reachable scales the volume stays order 1**: `delta^{-2}` tubes of radius `delta` in a box of
   volume 2.74 saturate the union toward a constant. The logarithmic bound is then verified
   **trivially** — the measured volume is orders of magnitude above `c/(1+log)^{66}`.
3. **The logarithmic shape and the exponent 66 are NOT observable here.** A notebook plotting
   `V . (1+log(1/delta))^{66}` and claiming to see the bound would be an artifact: three values of
   `delta`, all in the saturated regime.

Measurement by Monte-Carlo sampling (100,000 points), `delta = 0.10` then `0.05` — the size
`N = delta^{-2}` quadruples when `delta` halves, and cost follows: `delta = 0.01` requires
`N = 10,000` tubes, out of a notebook's reach.


In [6]:
# Mesure (b) : N = delta^-2 tubes, directions separees (repulsif) contre directions uniformes.
from itertools import combinations
rng = np.random.default_rng(20261024)

def directions_separees(N, delta, max_essais=4000):
    """Echantillonnage repulsif : on rejette une direction a moins de delta d'une direction gardee."""
    dirs = []
    for _ in range(N):
        for _ in range(max_essais):
            v = rng.normal(size=3); v /= np.linalg.norm(v)
            if all(np.linalg.norm(np.cross(v, u)) >= delta for u in dirs):
                dirs.append(v); break
        else:
            break
    return dirs

def separation_minimale(dirs):
    return min(np.linalg.norm(np.cross(a, b)) for a, b in combinations(dirs, 2))

def volume_union(dirs, delta, npts=100000, L=1.0):
    P = rng.uniform(-0.7, 0.7, size=(npts, 3))
    couvert = np.zeros(npts, dtype=bool)
    for u in dirs:
        d = np.linalg.norm(np.cross(P, u), axis=1)
        t = P @ u
        couvert |= (d <= delta) & (np.abs(t) <= L / 2)
    return couvert.mean() * 1.4 ** 3

for delta in (0.10, 0.05):
    N = int(round(1 / delta ** 2))
    dsep = directions_separees(N, delta)
    duni = [v / np.linalg.norm(v) for v in rng.normal(size=(N, 3))]
    print(f"delta={delta} N={N} (=delta^-2) | separation min : repulsif "
          f"{separation_minimale(dsep):.3f} >= delta, uniforme {separation_minimale(duni):.4f}")
    vsep = volume_union(dsep, delta)
    vuni = volume_union(duni, delta)
    print(f"   volume union : separe {vsep:.4f} | uniforme {vuni:.4f} "
          f"| boite 1.4^3 = {1.4**3:.3f} | 1+log(1/delta) = {1 + np.log(1/delta):.2f}")


delta=0.1 N=100 (=delta^-2) | separation min : repulsif 0.107 >= delta, uniforme 0.0124


   volume union : separe 0.5377 | uniforme 0.5152 | boite 1.4^3 = 2.744 | 1+log(1/delta) = 3.30


delta=0.05 N=400 (=delta^-2) | separation min : repulsif 0.050 >= delta, uniforme 0.0007


   volume union : separe 0.5230 | uniforme 0.4959 | boite 1.4^3 = 2.744 | 1+log(1/delta) = 4.00


**Verdict of measurement (b).** The three pre-registered points come out as announced:

1. **The premise is achievable and distinguishes the families**: the repulsive draw's minimal separation
   stays above `delta` (the requested constraint), while the uniform draw falls two orders of magnitude
   below. That is the quantity carrying the statement, and it is measured.
2. **The volume stays order 1** (~0.5 in a 2.74 box) at both scales: the regime is saturated, the bound
   `c/(1+log)^{66}` holds by a huge margin — **trivially**. This is not a confirmation of the logarithmic
   shape; it is the observation that we are far from its domain.
3. **The exponent 66 is not observable**, and not as a fixable instrument defect: three `delta` values in
   the saturated regime distinguish no exponent. The separated/uniform ratio (~0.02 to 0.03) is the only
   measurable gap between families, and it says nothing about the bound.

As for attributing the exponent: **66** belongs to the Lean development (`lean/docs/079.md`), not to the
manuscript — the 59-page PDF contains `66` only as a bibliographic reference number. The notebook says so
rather than fabricating a curve that would seem to measure it.


## 6. What we do not implement — and the measured false friends

Our lake carries Schwartz functions (`Calibration.Distribution`), and that is our only honest anchor. The
objects that would carry the rest of the theorem **do not exist** in our lakes:

- the **Bochner-Riesz operator** — mind the false friend: the only "Bochner" occurrence in our lakes is
  the Bochner **integral** (`Mathlib.MeasureTheory.Integral.Bochner.Basic`), unrelated;
- the **Kakeya maximal operator**;
- **Stein's square function**;
- the **Fourier restriction estimates** for the sphere, compact paraboloid and truncated cone.

The next cell measures these absences instead of asserting them — the only way to tell "we do not have
it" from "we did not look".


In [7]:
# Mesure des faux amis : chaque motif est cherche dans l'arbre, et le resultat est dit tel quel.
import subprocess

def grep_count(motif, chemin="../../"):
    r = subprocess.run(["git", "grep", "-il", motif, "--", chemin],
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
    fichiers = [l for l in (r.stdout or "").splitlines() if l.strip()]
    return fichiers

for motif in ("kakeya", "bochner", "restriction spherique", "stein square", "square function"):
    f = grep_count(motif)
    print(f"{motif:<22} {len(f):>3} fichier(s)" + (f"  ex. {f[0]}" if f else ""))
print()
print("Conclusion : aucune occurrence pertinente. Le seul 'bochner' est l'integrale de Bochner")
print("(MeasureTheory), pas le multiplicateur de Bochner-Riesz -- faux ami nomme, pas ancre.")


kakeya                   3 fichier(s)  ex. ANALYSE-13-Smoothing-Local-Kakeya.ipynb


bochner                  5 fichier(s)  ex. ANALYSE-13-Smoothing-Local-Kakeya.ipynb


restriction spherique    2 fichier(s)  ex. ANALYSE-13-Smoothing-Local-Kakeya.ipynb


stein square             2 fichier(s)  ex. ANALYSE-13-Smoothing-Local-Kakeya.ipynb


square function          2 fichier(s)  ex. ANALYSE-13-Smoothing-Local-Kakeya.ipynb

Conclusion : aucune occurrence pertinente. Le seul 'bochner' est l'integrale de Bochner
(MeasureTheory), pas le multiplicateur de Bochner-Riesz -- faux ami nomme, pas ancre.


## 7. Exercises

Three exercises, non-blocking: the notebook runs end to end even unsolved (rule C.1). None requires
external data — all three measure with what the notebook has already loaded.


### Exercise 1 — where is the critical point, and why `p_c = 2n/(n-1)`?

Compute `p_c` for `n = 4` and `n = 10`, then verify numerically that `s_+(p) = 0` exactly for
`p <= p_c` and strictly positive beyond.

> Hint: `s_+(p) = (n-1)/2 - n/p` vanishes when `p = 2n/(n-1)`. The `max(0, ...)` is not a writing
> convenience: it is the boundary between two physical regimes.


In [8]:
# Exercice 1 -- le point critique pour n = 4 et n = 10 (a completer)
# Etape 1 : calculer p_c = 2n/(n-1) pour n = 4 puis n = 10.
# Etape 2 : balayer p autour de p_c et verifier que s_plus(n, p) est nul en dessous,
#           strictement positif au dessus.
def point_critique(n):
    # TODO etudiant
    return None

print("Exercice a completer")


Exercice a completer


### Exercise 2 — the negative witness, turned around

Take the section 4 measurement again, but **remove** the time averaging: replace the integral over
`[1,2]` by a single evaluation at `T = 1.5`. The log-log slope should increase.

> Hint: this is already the "fixed time" column of measurement (a). Reproduce it using only `fixe` and
> plot both slopes side by side for `p = 4`, `n = 2`. To be settled by measurement: does the slope gap
> equal `1/p`, as the theorem suggests?


In [9]:
# Exercice 2 -- l'ecart des deux pentes vaut-il 1/p ? (a completer)
# Etape 1 : reprendre mesure(n=2, N=256, p=4, lam) pour lam in [8, 16, 32].
# Etape 2 : calculer les deux pentes (bande de temps / instant fixe) et leur difference.
# Etape 3 : comparer a 1/p et conclure en une phrase -- l'ecart mesure est-il celui annonce ?
def ecart_pentes(p=4, n=2, N=256):
    # TODO etudiant
    return None

print("Exercice a completer")


Exercice a completer


### Exercise 3 — a third family of tubes, nearly parallel

Build a family of tubes whose directions are separated by only `delta/4` (instead of `delta`), and
measure its union volume. Step outside the statement's hypothesis: what happens?

> Hint: repulsion is the mechanism. If the volume collapses toward the uniform-draw value, then the
> `delta` separation is not a technical detail of the statement — it is what produces the volume.


In [10]:
# Exercice 3 -- directions separees de delta/4 : le volume tient-il encore ? (a completer)
# Etape 1 : pour delta = 0.10 et N = delta^-2, construire directions_separees(N, delta/4).
# Etape 2 : mesurer separation_minimale et volume_union sur ce jeu, et comparer a la section 5.
# Etape 3 : dire en une phrase ce que la comparaison montre sur le role de la separation.
def tubes_presque_paralleles(delta=0.02):
    # TODO etudiant
    return None

print("Exercice a completer")


Exercice a completer


***

## What this notebook established, and what it did not

**Established with us, measurable at every run**: the numerical reading of the Sobolev order and its two
regimes; the effective existence of the Schwartz objects in our lake, probed by `#check` and not
redefined; two measurements with pre-registered hypotheses, with their negative witness; the measurement
of the false friends, telling "absent" from "not looked for".

**Not established, and quoted as such**: the proof itself. The six statements of the trust surface are
authors' statements; we neither import nor reconstruct them. The exponent `66` of the logarithmic bound is
neither the manuscript's nor measurable by this notebook.

**What this fold does not replace**: the manuscripts. They are in the shared repository, fingerprints in
the notebook header — that is where one reads what the proof does, not here.

Part of #19898 (openai/math digestion) — See #23142.
